# LeetCode 545 — Boundary of Binary Tree bằng Python

Link đề: https://leetcode.com/problems/boundary-of-binary-tree/ · Độ khó: **Medium** · Chủ đề: Tree, Depth-First Search, Binary Tree

Skeleton LeetCode:

```python
# Definition for a binary tree node.
# class TreeNode:
#     def __init__(self, val=0, left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def boundaryOfBinaryTree(self, root: Optional[TreeNode]) -> List[int]:
        ...
```

Ghi chú: đây là bài **Premium** (cần tài khoản trả phí trên LeetCode). Đề và các ví dụ được viết theo hiểu biết về bài, nên hãy **đối chiếu lại với đề gốc** nếu bạn có tài khoản Premium. Mọi đáp án trong notebook đã được kiểm chứng bằng code (so với một cách làm vét cạn độc lập và so giữa các cách). Các ví dụ ngoài hai ví dụ chính của đề là ví dụ **tự dựng** (đối chiếu lại đề gốc nếu cần). Notebook tự dựng `TreeNode` và hàm `build_tree` dựng cây từ list kiểu LeetCode (`[1,null,2,3,4]`).

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: **biên (boundary)** của cây nhị phân là ghép lại theo thứ tự **gốc**, **biên trái** (đi xuống từ con trái của gốc), **các lá** (từ trái sang phải) và **biên phải đảo ngược** (đi xuống ở phía phải rồi đọc từ dưới lên). Đây giống một vòng đi **ngược chiều kim đồng hồ** quanh cây.
2. Nắm các công cụ Python: định nghĩa `class TreeNode`, dựng cây từ list có `None`, vẽ cây bằng chữ, DFS đệ quy và **ngăn xếp tường minh**, biểu thức điều kiện `a if cond else b`, `reversed` / cắt `[::-1]`, giới hạn đệ quy của Python.
3. Đi từ **kiểm tra từng nút bằng đường đi từ gốc (`O(n·h)`)** → **ba phần: đi dọc biên trái, thu lá, đi dọc biên phải (`O(n)`, không đệ quy)** → **một hàm đệ quy với "vai trò" `0/1/2`** → **một lượt DFS với hai cờ `lb`/`rb` và mẹo hậu tự để tự đảo biên phải**.
4. Hiểu mấu chốt: biên trái luôn **ưu tiên con trái, nếu không có mới rẽ sang con phải**; biên phải ngược lại; **lá không thuộc biên trái/phải** (nó đã nằm trong nhóm "lá"); gốc không phải lá và chỉ được liệt kê **một lần**.
5. Tránh các lỗi kinh điển: tính **lá hai lần**, quên rẽ sang con phải khi biên trái không có con trái, quên **đảo** biên phải, coi nút chỉ có một con là lá, và quên xử lý cây chỉ có một nút.

## 2. Đề bài & ví dụ

**Biên (boundary)** của một cây nhị phân là phép nối theo thứ tự: **gốc**, **biên trái**, **các lá** sắp từ trái sang phải, và **biên phải theo thứ tự ngược**.

**Biên trái** là tập các nút xác định như sau:

- Con trái của gốc thuộc biên trái. Nếu gốc không có con trái thì biên trái **rỗng**.
- Nếu một nút thuộc biên trái và **có con trái** thì con trái của nó thuộc biên trái.
- Nếu một nút thuộc biên trái, **không có con trái** nhưng **có con phải** thì con phải của nó thuộc biên trái.
- **Lá trái nhất không thuộc** biên trái.

**Biên phải** tương tự biên trái nhưng ở phía **con phải của gốc** (ưu tiên con phải, nếu không có thì rẽ sang con trái); lá cũng **không** thuộc biên phải; nếu gốc không có con phải thì biên phải rỗng.

**Lá** là nút không có con. Với bài này **gốc không được coi là lá**.

Cho `root` của cây nhị phân, trả về **các giá trị trên biên** của nó.

**Ví dụ 1**

- Input: `root = [1,null,2,3,4]`
- Output: `[1,3,4,2]`
- Giải thích: gốc không có con trái nên biên trái rỗng. Biên phải đi từ con phải `2` xuống `4`; `4` là lá nên biên phải là `[2]`. Các lá từ trái sang phải: `[3,4]`. Ghép lại: `[1] + [] + [3,4] + [2] = [1,3,4,2]`.

**Ví dụ 2**

- Input: `root = [1,2,3,4,5,6,null,null,null,7,8,9,10]`
- Output: `[1,2,4,7,8,9,10,6,3]`
- Giải thích: biên trái đi `2 -> 4`, `4` là lá nên biên trái là `[2]`. Biên phải đi `3 -> 6 -> 10`, `10` là lá nên biên phải là `[3,6]`, đảo lại thành `[6,3]`. Các lá: `[4,7,8,9,10]`. Ghép: `[1] + [2] + [4,7,8,9,10] + [6,3]`.

**Ràng buộc:**

- Số nút trong cây nằm trong khoảng `[1, 10^4]`.
- `-1000 <= Node.val <= 1000`

Chú ý: số nút có thể lên tới `10^4`, nên cây có thể **rất sâu** (một "con đường thẳng" dài `10^4`). Giá trị có thể **trùng nhau**, vì vậy ta phải xét theo **nút** chứ không theo **giá trị**.

In [1]:
from typing import List, Optional
from collections import deque
import random
import sys


class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right


def build_tree(values):
    """Dựng cây từ list kiểu LeetCode (duyệt theo tầng, None là nút rỗng)."""
    if not values or values[0] is None:
        return None
    root = TreeNode(values[0])
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        if i < len(values) and values[i] is not None:
            node.left = TreeNode(values[i])
            queue.append(node.left)
        i += 1
        if i < len(values) and values[i] is not None:
            node.right = TreeNode(values[i])
            queue.append(node.right)
        i += 1
    return root


def tree_to_list(root):
    """Ngược lại: cây -> list kiểu LeetCode (cắt None thừa ở cuối)."""
    if root is None:
        return []
    out, queue = [], deque([root])
    while queue:
        node = queue.popleft()
        if node is None:
            out.append(None)
        else:
            out.append(node.val)
            queue.append(node.left)
            queue.append(node.right)
    while out and out[-1] is None:
        out.pop()
    return out


ex1 = [1, None, 2, 3, 4]
ex2 = [1, 2, 3, 4, 5, 6, None, None, None, 7, 8, 9, 10]
examples = [
    (ex1, [1, 3, 4, 2]),
    (ex2, [1, 2, 4, 7, 8, 9, 10, 6, 3]),
]
for vals, expected in examples:
    assert tree_to_list(build_tree(vals)) == vals
    print(vals, "-> kỳ vọng", expected)

[1, None, 2, 3, 4] -> kỳ vọng [1, 3, 4, 2]
[1, 2, 3, 4, 5, 6, None, None, None, 7, 8, 9, 10] -> kỳ vọng [1, 2, 4, 7, 8, 9, 10, 6, 3]


## 3. Kiến thức nền

### 3.1. Nút cây, cây nhị phân và "lá"

Mỗi nút (`TreeNode`) có `val` (giá trị), `left`, `right` (nút con hoặc `None`). Chỉ cần giữ nút gốc là truy cập được cả cây. Một nút là **lá** nếu **cả hai** con đều là `None`: `node.left is None and node.right is None`. Chú ý: nút chỉ có **một** con **không phải** lá.

Trong Python, `None` là "giả" (falsy) còn một đối tượng `TreeNode` bình thường là "thật" (truthy), nên `if node.left:` nghĩa là "có con trái".

In [2]:
a = TreeNode(1, TreeNode(2), None)      # gốc 1 chỉ có con trái 2
b = a.left


def is_leaf(node):
    return node.left is None and node.right is None


print(is_leaf(a), is_leaf(b))            # False True
print(bool(a.left), bool(a.right))       # True False

False True
True False


### 3.2. Dựng cây từ list kiểu LeetCode và vẽ cây bằng chữ

List `[1,2,3,4,5,6,None,None,None,7,8,9,10]` được đọc **theo từng tầng, từ trái sang phải**; `None` là "không có nút" (và không có con). Hàm `build_tree` dùng hàng đợi: mỗi nút lấy ra nhận hai phần tử kế tiếp làm con trái, con phải.

Để **nhìn** được cây, ta viết `draw(root)`, in mỗi nút trên một dòng; `L:` là con trái, `R:` là con phải.

In [3]:
def draw(root):
    """In cây bằng chữ: L = con trái, R = con phải."""
    if root is None:
        print("(cây rỗng)")
        return
    print(root.val)

    def go(node, prefix):
        kids = [(tag, k) for tag, k in (("L", node.left), ("R", node.right)) if k is not None]
        for idx, (tag, kid) in enumerate(kids):
            last = idx == len(kids) - 1
            print(prefix + ("└─" if last else "├─") + tag + ":" + str(kid.val))
            go(kid, prefix + ("   " if last else "│  "))

    go(root, "")


tree2 = build_tree(ex2)
draw(tree2)

1
├─L:2
│  ├─L:4
│  └─R:5
│     ├─L:7
│     └─R:8
└─R:3
   └─L:6
      ├─L:9
      └─R:10


### 3.3. DFS đệ quy và thứ tự tiền tự (preorder)

DFS (duyệt theo chiều sâu) làm việc với nút hiện tại rồi gọi đệ quy cho con trái, con phải. Gọi cho `None` thì trả về ngay (**điều kiện dừng**). Thứ tự **tiền tự** là: **nút → trái → phải**. Với mỗi nút ta có hai thời điểm: **khi vào** (trước khi duyệt con) và **khi ra** (sau khi duyệt xong hai con). Mẹo ở Cách 4 dùng thời điểm **ra** để tự đảo thứ tự.

In [4]:
def preorder_demo(node, depth=0):
    if node is None:
        return
    print("  " * depth + f"vào {node.val}")
    preorder_demo(node.left, depth + 1)
    preorder_demo(node.right, depth + 1)
    print("  " * depth + f"ra  {node.val}")


preorder_demo(build_tree([1, 2, 3]))

vào 1
  vào 2
  ra  2
  vào 3
  ra  3
ra  1


### 3.4. DFS không đệ quy bằng ngăn xếp (stack)

Một `list` dùng `append` (đẩy) và `pop()` (lấy phần tử cuối) hoạt động như **ngăn xếp**. Để duyệt tiền tự **trái trước phải sau** bằng ngăn xếp: lấy một nút ra, xử lý nó, rồi **đẩy con phải trước, con trái sau** (để con trái được lấy ra trước). Ưu điểm: **không bị giới hạn độ sâu đệ quy**.

In [5]:
def preorder_stack(root):
    out, stack = [], [root]
    while stack:
        node = stack.pop()
        out.append(node.val)
        if node.right:
            stack.append(node.right)
        if node.left:
            stack.append(node.left)
    return out


print(preorder_stack(tree2))     # [1, 2, 4, 5, 7, 8, 3, 6, 9, 10]

[1, 2, 4, 5, 7, 8, 3, 6, 9, 10]


### 3.5. Biểu thức điều kiện `a if cond else b`, `reversed` và `[::-1]`

- `x if cond else y` chọn giá trị: ví dụ `node.left if node.left else node.right` nghĩa là "con trái nếu có, không thì con phải", đúng quy tắc **rẽ** của biên trái.
- Đảo list: `lst[::-1]` tạo list mới đã đảo; `reversed(lst)` cho iterator đảo; `lst.reverse()` đảo ngay tại chỗ. `a.extend(b)` nối `b` vào cuối `a`.

In [6]:
lst = [3, 6]
print(lst[::-1], list(reversed(lst)))     # [6, 3] [6, 3]
ans = [1, 2]
ans.extend(reversed([8, 9]))
print(ans)                                # [1, 2, 9, 8]

node = TreeNode(5, None, TreeNode(7))     # chỉ có con phải
nxt = node.left if node.left else node.right
print(nxt.val)                            # 7  (rẽ sang phải vì không có con trái)

[6, 3] [6, 3]
[1, 2, 9, 8]
7


### 3.6. Giới hạn đệ quy của Python

Python mặc định chỉ cho đệ quy sâu khoảng `1000` tầng (`sys.getrecursionlimit()`). Cây có tới `10^4` nút có thể là một đường thẳng sâu `10^4` tầng, nên **đệ quy thuần có thể báo `RecursionError`** nếu máy chấm không nới giới hạn. Vì vậy ta sẽ chọn cách **không đệ quy** (vòng `while` + ngăn xếp) cho code nộp, đồng thời vẫn học các cách đệ quy.

In [7]:
print(sys.getrecursionlimit())     # thường là 1000


def depth_recursive(k):
    return 0 if k == 0 else 1 + depth_recursive(k - 1)


try:
    depth_recursive(5000)
except RecursionError:
    print("RecursionError: đệ quy sâu 5000 vượt giới hạn mặc định")

1000
RecursionError: đệ quy sâu 5000 vượt giới hạn mặc định


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy tưởng tượng bạn **đi một vòng quanh cây**, bám sát mép ngoài, **ngược chiều kim đồng hồ**, bắt đầu từ gốc. Bạn sẽ đi qua:

1. **Gốc.**
2. **Mép trái** đi xuống: những nút ở sườn trái.
3. **Đáy**: lần lượt từng **lá** từ trái sang phải.
4. **Mép phải** đi **lên**: những nút ở sườn phải, nhưng đọc **từ dưới lên trên**.

Với cây của ví dụ 2:

In [8]:
draw(tree2)

1
├─L:2
│  ├─L:4
│  └─R:5
│     ├─L:7
│     └─R:8
└─R:3
   └─L:6
      ├─L:9
      └─R:10


Tính tay từng phần:

| Phần | Cách tìm | Kết quả |
|---|---|---|
| Gốc | luôn là phần tử đầu | `[1]` |
| Biên trái | bắt đầu ở `2` (con trái của gốc): `2` không phải lá → lấy; `2` có con trái `4` → sang `4`; `4` là lá → **không lấy** | `[2]` |
| Lá (trái → phải) | các nút không có con: `4`, `7`, `8`, `9`, `10` | `[4,7,8,9,10]` |
| Biên phải | bắt đầu ở `3` (con phải của gốc): lấy `3`; `3` không có con phải → rẽ trái sang `6`; `6` không phải lá → lấy; `6` có con phải `10` → sang `10`; `10` là lá → **không lấy**. Được `[3,6]` rồi **đảo** | `[6,3]` |
| Ghép | `[1] + [2] + [4,7,8,9,10] + [6,3]` | `[1,2,4,7,8,9,10,6,3]` |

Ba chỗ dễ nhầm rút ra ngay từ ví dụ này: (1) lá `4` nằm ở **cuối biên trái** nhưng **chỉ liệt kê trong nhóm lá**; (2) biên phải rẽ sang **con trái** khi **không có con phải** (`3 → 6`); (3) biên phải được **viết từ trên xuống rồi đảo** (hoặc ghi sau).

In [9]:
# Kiểm tra bằng tay cho ví dụ 2
root = build_tree(ex2)
left_part = [2]
leaves = [4, 7, 8, 9, 10]
right_part_top_down = [3, 6]
manual = [root.val] + left_part + leaves + right_part_top_down[::-1]
print(manual)
assert manual == examples[1][1]

[1, 2, 4, 7, 8, 9, 10, 6, 3]


## 5. Cách 1 — Brute Force / baseline: kiểm tra từng nút bằng đường đi từ gốc

**Ý tưởng.** Thay vì "đi dọc biên", ta **hỏi từng nút** "bạn có thuộc biên trái không? biên phải không? có là lá không?". Để trả lời, ta cần **đường đi từ gốc tới nút đó** (`path`, một list các nút).

- Nút `x` **thuộc biên trái** nếu `x` không phải lá và đường đi `path = [root, p1, p2, ..., x]` thỏa: `p1` chính là `root.left`; và với mỗi `k ≥ 1`, bước tiếp theo `p(k+1)` đúng là **con trái của `p(k)` nếu nó tồn tại, không thì con phải**.
- Tương tự cho biên phải (ưu tiên con phải).
- Nút `x` **là lá** nếu không có con và `x` không phải gốc.

Duyệt tiền tự cả cây, giữ `path` bằng `append`/`pop`; với mỗi nút chạy hàm kiểm tra tốn `O(độ sâu)`. Thứ tự tiền tự cho ta: biên trái từ **trên xuống**, lá từ **trái sang phải**, biên phải từ **trên xuống** (nên cuối cùng đảo lại).

In [10]:
def on_edge(path, prefer_left):
    """path = [root, ..., x]. x có nằm trên biên (trái nếu prefer_left else phải) không?
    (chưa xét x có phải lá hay không)"""
    if len(path) < 2:
        return False
    first = path[0].left if prefer_left else path[0].right
    if path[1] is not first:
        return False
    for k in range(1, len(path) - 1):
        cur = path[k]
        if prefer_left:
            expected = cur.left if cur.left is not None else cur.right
        else:
            expected = cur.right if cur.right is not None else cur.left
        if path[k + 1] is not expected:
            return False
    return True


def boundary_bruteforce(root):
    """Cách 1: kiểm tra từng nút bằng đường đi từ gốc. Time O(n * h)."""
    left, leaves, right = [], [], []
    path = []

    def visit(node):
        path.append(node)
        leaf = node.left is None and node.right is None
        if node is not root:
            if leaf:
                leaves.append(node.val)
            elif on_edge(path, True):
                left.append(node.val)
            elif on_edge(path, False):
                right.append(node.val)
        if node.left:
            visit(node.left)
        if node.right:
            visit(node.right)
        path.pop()

    visit(root)
    return [root.val] + left + leaves + right[::-1]


for vals, expected in examples:
    got = boundary_bruteforce(build_tree(vals))
    print(vals, "->", got, got == expected)

[1, None, 2, 3, 4] -> [1, 3, 4, 2] True
[1, 2, 3, 4, 5, 6, None, None, None, 7, 8, 9, 10] -> [1, 2, 4, 7, 8, 9, 10, 6, 3] True


Để thấy rõ "mỗi nút bị phân loại thế nào", ta in vai trò của từng nút trong ví dụ 2:

In [11]:
def classify_all(root):
    path = []

    def visit(node):
        path.append(node)
        leaf = node.left is None and node.right is None
        if node is root:
            role = "gốc"
        elif leaf:
            role = "lá"
        elif on_edge(path, True):
            role = "biên trái"
        elif on_edge(path, False):
            role = "biên phải"
        else:
            role = "nút trong (không thuộc biên)"
        print(f"nút {node.val:>2} | đường đi {[p.val for p in path]} | {role}")
        if node.left:
            visit(node.left)
        if node.right:
            visit(node.right)
        path.pop()

    visit(root)


classify_all(tree2)

nút  1 | đường đi [1] | gốc
nút  2 | đường đi [1, 2] | biên trái
nút  4 | đường đi [1, 2, 4] | lá
nút  5 | đường đi [1, 2, 5] | nút trong (không thuộc biên)
nút  7 | đường đi [1, 2, 5, 7] | lá
nút  8 | đường đi [1, 2, 5, 8] | lá
nút  3 | đường đi [1, 3] | biên phải
nút  6 | đường đi [1, 3, 6] | biên phải
nút  9 | đường đi [1, 3, 6, 9] | lá
nút 10 | đường đi [1, 3, 6, 10] | lá


**Độ phức tạp.** Mỗi nút gọi `on_edge` duyệt hết `path` (độ dài tới `h`), nên Time `O(n · h)`: tốt nhất cây cân bằng `O(n log n)`, **xấu nhất cây là một đường thẳng `O(n²)`** (với `n = 10^4` là khoảng `5·10^7` bước so sánh, rất chậm trong Python). Space `O(h)` cho `path` và ngăn xếp đệ quy, ngoài ra còn kết quả.

**Nút thắt.** Với mỗi nút ta **kiểm tra lại cả đường đi** từ gốc, trong khi biên là một **đường đi liên tục** mà ta có thể **đi theo một lần** từ trên xuống. Không cần "hỏi" nữa, chỉ cần "đi".

## 6. Ý tưởng thuật toán / bước đột phá

Nhìn biên như **ba phần độc lập**, mỗi phần là một việc đơn giản:

1. **Biên trái**: bắt đầu ở `root.left`, mỗi bước đi xuống **con trái nếu có, không thì con phải**, thu giá trị của nút **không phải lá**. Đây chỉ là **một vòng `while`**, không cần đệ quy.
2. **Biên phải**: đối xứng, ưu tiên **con phải**. Thu từ trên xuống rồi **đảo** (hoặc `reversed` khi nối).
3. **Lá**: duyệt cây theo **tiền tự (trái trước phải sau)** và thu mọi nút không có con; thứ tự này tự cho **lá từ trái sang phải**. Không đệ quy: dùng ngăn xếp.

Gốc xử lý riêng: nếu cây chỉ có **một nút** thì đáp án là `[root.val]` (gốc không phải lá, không lặp). Còn lại, gốc luôn đứng đầu.

**Vì sao không đụng nhau?** Biên trái, biên phải **không chứa lá** nên không trùng nhóm lá; chúng ở hai cây con khác nhau của gốc nên không trùng nhau; và gốc được thêm riêng. Do đó **mỗi nút xuất hiện tối đa một lần**.

Có thêm hai hướng khác:

- **Một hàm đệ quy dùng "vai trò" `i = 0/1/2`** cho ba phần (Cách 3), tương đương Cách 2 nhưng viết bằng đệ quy.
- **Một lượt DFS duy nhất** mang theo hai cờ `lb`, `rb` ("tôi đang ở trên biên trái/phải"); biên trái ghi khi **vào** nút (tiền tự), biên phải ghi khi **ra** (hậu tự, tự đảo thứ tự) (Cách 4).

## 7. Các cách tiếp theo

### Cách 2 — Ba phần bằng vòng `while` và ngăn xếp (không đệ quy)

**Giải thích từng đoạn.**

- `if not root.left and not root.right: return [root.val]`: cây một nút.
- `ans = [root.val]`: gốc luôn đứng đầu.
- **Biên trái:** `node = root.left`; `while node:` nếu `node` không phải lá thì `ans.append(node.val)`; rồi `node = node.left if node.left else node.right`. Khi `node` là lá, ta không thêm; sau bước rẽ `node` thành `None` và vòng lặp kết thúc.
- **Lá:** ngăn xếp `stack = [root]`. Lấy `node`; nếu là lá thì `ans.append(node.val)`; không thì đẩy **con phải trước, con trái sau**. (Gốc có con nên không bị coi là lá.)
- **Biên phải:** như biên trái nhưng ưu tiên con phải, thu vào list tạm `right`, rồi `ans.extend(reversed(right))`.

In [12]:
def boundary_walk_demo(root):
    if root.left is None and root.right is None:
        return [root.val]
    ans = [root.val]
    print("gốc:", ans)

    node = root.left
    while node:
        if node.left or node.right:
            ans.append(node.val)
            print("  biên trái lấy", node.val)
        else:
            print("  biên trái gặp lá", node.val, "-> bỏ, dừng")
        node = node.left if node.left else node.right
    print("sau biên trái:", ans)

    stack = [root]
    while stack:
        node = stack.pop()
        if node.left is None and node.right is None:
            ans.append(node.val)
            print("  lá", node.val)
        else:
            if node.right:
                stack.append(node.right)
            if node.left:
                stack.append(node.left)
    print("sau lá:", ans)

    right = []
    node = root.right
    while node:
        if node.left or node.right:
            right.append(node.val)
            print("  biên phải lấy", node.val)
        else:
            print("  biên phải gặp lá", node.val, "-> bỏ, dừng")
        node = node.right if node.right else node.left
    ans.extend(reversed(right))
    print("sau biên phải (đã đảo):", ans)
    return ans


boundary_walk_demo(tree2)

gốc: [1]
  biên trái lấy 2
  biên trái gặp lá 4 -> bỏ, dừng
sau biên trái: [1, 2]
  lá 4
  lá 7
  lá 8
  lá 9
  lá 10
sau lá: [1, 2, 4, 7, 8, 9, 10]
  biên phải lấy 3
  biên phải lấy 6
  biên phải gặp lá 10 -> bỏ, dừng
sau biên phải (đã đảo): [1, 2, 4, 7, 8, 9, 10, 6, 3]


[1, 2, 4, 7, 8, 9, 10, 6, 3]

In [13]:
class Solution2:
    def boundaryOfBinaryTree(self, root: Optional[TreeNode]) -> List[int]:
        if not root.left and not root.right:
            return [root.val]
        ans = [root.val]

        node = root.left                      # biên trái (từ trên xuống)
        while node:
            if node.left or node.right:
                ans.append(node.val)
            node = node.left if node.left else node.right

        stack = [root]                        # các lá, trái -> phải
        while stack:
            node = stack.pop()
            if not node.left and not node.right:
                ans.append(node.val)
            else:
                if node.right:
                    stack.append(node.right)
                if node.left:
                    stack.append(node.left)

        right = []                            # biên phải (từ trên xuống), rồi đảo
        node = root.right
        while node:
            if node.left or node.right:
                right.append(node.val)
            node = node.right if node.right else node.left
        ans.extend(reversed(right))
        return ans


for vals, expected in examples:
    assert Solution2().boundaryOfBinaryTree(build_tree(vals)) == expected
print("Cách 2 đúng trên 2 ví dụ đề")

Cách 2 đúng trên 2 ví dụ đề


**Độ phức tạp Cách 2:** mỗi phần chạm mỗi nút tối đa một lần ⇒ Time `O(n)`. Space: ngăn xếp tối đa `O(h)` (mỗi tầng đẩy tối đa thêm một nút treo), list `right` tối đa `O(h)`, cộng kết quả `O(n)`. **Không có đệ quy** nên an toàn với cây sâu `10^4`.

---

### Cách 3 — Một hàm đệ quy với "vai trò" `i = 0 / 1 / 2`

**Giải thích từng đoạn.** Viết một hàm `dfs(nums, node, i)` thu giá trị vào `nums` theo vai trò `i`:

- `i = 0` (biên trái): nếu `node` không phải lá thì thu `node.val` rồi đệ quy vào **con trái nếu có, không thì con phải** (vẫn vai trò `0`). Nếu là lá thì dừng, không thu.
- `i = 1` (lá): nếu là lá thì thu; ngược lại đệ quy sang **cả hai con** (vai trò `1`).
- `i = 2` (biên phải): như vai trò `0` nhưng ưu tiên con phải.

Gọi `dfs(left, root.left, 0)`, `dfs(leaves, root, 1)`, `dfs(right, root.right, 2)` rồi ghép `[root.val] + left + leaves + right[::-1]`. Đây chính là hướng giải chính thức, đúng tinh thần Cách 2 nhưng bằng đệ quy.

In [14]:
class Solution3:
    def boundaryOfBinaryTree(self, root: Optional[TreeNode]) -> List[int]:
        def is_leaf(node):
            return node.left is None and node.right is None

        def dfs(nums, node, i):
            if node is None:
                return
            if i == 0:                                # biên trái
                if not is_leaf(node):
                    nums.append(node.val)
                    dfs(nums, node.left if node.left else node.right, 0)
            elif i == 1:                              # lá
                if is_leaf(node):
                    nums.append(node.val)
                else:
                    dfs(nums, node.left, 1)
                    dfs(nums, node.right, 1)
            else:                                     # biên phải
                if not is_leaf(node):
                    nums.append(node.val)
                    dfs(nums, node.right if node.right else node.left, 2)

        if is_leaf(root):
            return [root.val]
        left, leaves, right = [], [], []
        dfs(left, root.left, 0)
        dfs(leaves, root, 1)
        dfs(right, root.right, 2)
        return [root.val] + left + leaves + right[::-1]


for vals, expected in examples:
    assert Solution3().boundaryOfBinaryTree(build_tree(vals)) == expected
print("Cách 3 đúng trên 2 ví dụ đề")

Cách 3 đúng trên 2 ví dụ đề


**Độ phức tạp Cách 3:** Time `O(n)` (ba lượt, mỗi lượt không quá `n` nút), Space `O(h)` cho ngăn xếp đệ quy cộng kết quả `O(n)`. **Lưu ý:** cần độ sâu đệ quy tới `h`, trong Python cây sâu `10^4` có thể chạm giới hạn mặc định (xem 3.6).

---

### Cách 4 — Một lượt DFS với hai cờ `lb`, `rb` và mẹo hậu tự

**Ý tưởng.** Đi xuống mang theo hai cờ cho mỗi nút (không tính gốc):

- `lb` (left boundary): "nút này đang ở trên biên trái". Con trái **thừa hưởng** cờ `lb` của cha. Con phải thừa hưởng `lb` **chỉ khi cha không có con trái** (đúng quy tắc rẽ).
- `rb` đối xứng: con phải thừa hưởng `rb` của cha; con trái thừa hưởng `rb` chỉ khi cha **không có con phải**.

Quy tắc ghi giá trị:

- **Khi vào nút** (tiền tự): nếu nút là **lá** hoặc `lb` đang bật thì ghi ngay. Biên trái nằm trên một đường từ trên xuống nên thứ tự tiền tự cho đúng thứ tự "gốc → xuống"; các lá thì tiền tự cho đúng thứ tự trái → phải; và mọi biên trái đều xuất hiện **trước** các lá (xem mục 10).
- **Khi ra khỏi nút** (hậu tự): nếu `rb` bật và nút **không phải lá** thì ghi. Vì hậu tự **ra từ sâu lên nông**, ta nhận được biên phải **đã đảo** mà không cần `reverse`.

Gốc xử lý riêng: ghi `root.val`, rồi gọi `dfs(root.left, True, False)` và `dfs(root.right, False, True)`.

In [15]:
def trace_oneshot(root):
    ans = [root.val]
    print("gốc:", ans)
    if root.left is None and root.right is None:
        return ans

    def dfs(node, lb, rb, depth):
        if node is None:
            return
        leaf = node.left is None and node.right is None
        pad = "  " * depth
        if leaf or lb:
            ans.append(node.val)
        print(f"{pad}vào {node.val}: lb={lb!s:5} rb={rb!s:5} lá={leaf!s:5} ans={ans}")
        dfs(node.left, lb, rb and node.right is None, depth + 1)
        dfs(node.right, lb and node.left is None, rb, depth + 1)
        if rb and not leaf:
            ans.append(node.val)
            print(f"{pad}ra  {node.val}: ghi biên phải (hậu tự) ans={ans}")

    dfs(root.left, True, False, 0)
    dfs(root.right, False, True, 0)
    return ans


print(trace_oneshot(tree2))

gốc: [1]
vào 2: lb=True  rb=False lá=False ans=[1, 2]
  vào 4: lb=True  rb=False lá=True  ans=[1, 2, 4]
  vào 5: lb=False rb=False lá=False ans=[1, 2, 4]
    vào 7: lb=False rb=False lá=True  ans=[1, 2, 4, 7]
    vào 8: lb=False rb=False lá=True  ans=[1, 2, 4, 7, 8]
vào 3: lb=False rb=True  lá=False ans=[1, 2, 4, 7, 8]
  vào 6: lb=False rb=True  lá=False ans=[1, 2, 4, 7, 8]
    vào 9: lb=False rb=False lá=True  ans=[1, 2, 4, 7, 8, 9]
    vào 10: lb=False rb=True  lá=True  ans=[1, 2, 4, 7, 8, 9, 10]
  ra  6: ghi biên phải (hậu tự) ans=[1, 2, 4, 7, 8, 9, 10, 6]
ra  3: ghi biên phải (hậu tự) ans=[1, 2, 4, 7, 8, 9, 10, 6, 3]
[1, 2, 4, 7, 8, 9, 10, 6, 3]


In [16]:
class Solution4:
    def boundaryOfBinaryTree(self, root: Optional[TreeNode]) -> List[int]:
        ans = [root.val]
        if not root.left and not root.right:
            return ans

        def dfs(node, lb, rb):
            if node is None:
                return
            leaf = not node.left and not node.right
            if leaf or lb:                    # vào: biên trái hoặc lá (tiền tự)
                ans.append(node.val)
            dfs(node.left, lb, rb and node.right is None)
            dfs(node.right, lb and node.left is None, rb)
            if rb and not leaf:               # ra: biên phải (hậu tự => đã đảo)
                ans.append(node.val)

        dfs(root.left, True, False)
        dfs(root.right, False, True)
        return ans


for vals, expected in examples:
    assert Solution4().boundaryOfBinaryTree(build_tree(vals)) == expected
print("Cách 4 đúng trên 2 ví dụ đề")

Cách 4 đúng trên 2 ví dụ đề


**Độ phức tạp Cách 4:** một lượt DFS ⇒ Time `O(n)`, Space `O(h)` ngăn xếp đệ quy cộng kết quả `O(n)`. Ưu điểm: một hàm, một lượt, không cần list phụ; nhược điểm: khó đọc hơn và cũng dùng đệ quy.

## 8. TRACE từng bước

**Bảng trạng thái cho Cách 2 với ví dụ 2** (`[1,2,3,4,5,6,null,null,null,7,8,9,10]`):

| Giai đoạn | Nút đang xét | Có phải lá? | Hành động | `ans` / `right` |
|---|---|---|---|---|
| gốc | `1` | không | thêm gốc | `ans = [1]` |
| biên trái | `2` | không | thêm; rẽ trái sang `4` | `ans = [1,2]` |
| biên trái | `4` | **lá** | bỏ qua; kết thúc | `ans = [1,2]` |
| lá (stack) | pop `1` | không | đẩy `3`, rồi `2` | |
| lá | pop `2` | không | đẩy `5`, rồi `4` | |
| lá | pop `4` | lá | thêm | `ans = [1,2,4]` |
| lá | pop `5` | không | đẩy `8`, rồi `7` | |
| lá | pop `7`, `8` | lá | thêm `7`, `8` | `ans = [1,2,4,7,8]` |
| lá | pop `3` | không | đẩy `6` (không có con phải) | |
| lá | pop `6` | không | đẩy `10`, rồi `9` | |
| lá | pop `9`, `10` | lá | thêm | `ans = [1,2,4,7,8,9,10]` |
| biên phải | `3` | không | thêm vào `right`; không có con phải nên rẽ trái sang `6` | `right = [3]` |
| biên phải | `6` | không | thêm; rẽ phải sang `10` | `right = [3,6]` |
| biên phải | `10` | **lá** | bỏ qua; kết thúc | |
| ghép | | | `ans.extend(reversed(right))` | `ans = [1,2,4,7,8,9,10,6,3]` |

Hàm `trace_boundary` dưới đây in lại đúng các bước đó (bao gồm trạng thái ngăn xếp).

In [17]:
def trace_boundary(root):
    if root.left is None and root.right is None:
        print("cây một nút")
        return [root.val]
    ans = [root.val]
    print("[gốc] ans =", ans)

    print("[biên trái]")
    node = root.left
    while node:
        leaf = node.left is None and node.right is None
        if leaf:
            print(f"  nút {node.val}: lá -> bỏ, dừng")
        else:
            ans.append(node.val)
            nxt = node.left if node.left else node.right
            print(f"  nút {node.val}: lấy; rẽ sang {nxt.val} (ans = {ans})")
        node = node.left if node.left else node.right

    print("[lá]")
    stack = [root]
    while stack:
        print("  stack =", [x.val for x in stack])
        node = stack.pop()
        if node.left is None and node.right is None:
            ans.append(node.val)
            print(f"  pop {node.val}: lá -> thêm (ans = {ans})")
        else:
            if node.right:
                stack.append(node.right)
            if node.left:
                stack.append(node.left)
            print(f"  pop {node.val}: không phải lá -> đẩy con phải rồi con trái")

    print("[biên phải]")
    right = []
    node = root.right
    while node:
        leaf = node.left is None and node.right is None
        if leaf:
            print(f"  nút {node.val}: lá -> bỏ, dừng")
        else:
            right.append(node.val)
            print(f"  nút {node.val}: lấy (right = {right})")
        node = node.right if node.right else node.left
    ans.extend(reversed(right))
    print("[ghép] ans =", ans)
    return ans


assert trace_boundary(tree2) == examples[1][1]

[gốc] ans = [1]
[biên trái]
  nút 2: lấy; rẽ sang 4 (ans = [1, 2])
  nút 4: lá -> bỏ, dừng
[lá]
  stack = [1]
  pop 1: không phải lá -> đẩy con phải rồi con trái
  stack = [3, 2]
  pop 2: không phải lá -> đẩy con phải rồi con trái
  stack = [3, 5, 4]
  pop 4: lá -> thêm (ans = [1, 2, 4])
  stack = [3, 5]
  pop 5: không phải lá -> đẩy con phải rồi con trái
  stack = [3, 8, 7]
  pop 7: lá -> thêm (ans = [1, 2, 4, 7])
  stack = [3, 8]
  pop 8: lá -> thêm (ans = [1, 2, 4, 7, 8])
  stack = [3]
  pop 3: không phải lá -> đẩy con phải rồi con trái
  stack = [6]
  pop 6: không phải lá -> đẩy con phải rồi con trái
  stack = [10, 9]
  pop 9: lá -> thêm (ans = [1, 2, 4, 7, 8, 9])
  stack = [10]
  pop 10: lá -> thêm (ans = [1, 2, 4, 7, 8, 9, 10])
[biên phải]
  nút 3: lấy (right = [3])
  nút 6: lấy (right = [3, 6])
  nút 10: lá -> bỏ, dừng
[ghép] ans = [1, 2, 4, 7, 8, 9, 10, 6, 3]


**Trace nhanh cho ví dụ 1** (`[1,null,2,3,4]`): gốc `1` ⇒ `[1]`. Biên trái: `root.left` là `None` ⇒ rỗng. Lá: `1` (có con) → đẩy `2`; `2` có con → đẩy `4`, `3`; pop `3` lá, pop `4` lá ⇒ `[1,3,4]`. Biên phải: `2` không phải lá ⇒ `right = [2]`; rẽ phải sang `4` là lá ⇒ dừng. Ghép: `[1,3,4,2]`.

In [18]:
tree1 = build_tree(ex1)
draw(tree1)
assert trace_boundary(tree1) == [1, 3, 4, 2]

1
└─R:2
   ├─L:3
   └─R:4
[gốc] ans = [1]
[biên trái]
[lá]
  stack = [1]
  pop 1: không phải lá -> đẩy con phải rồi con trái
  stack = [2]
  pop 2: không phải lá -> đẩy con phải rồi con trái
  stack = [4, 3]
  pop 3: lá -> thêm (ans = [1, 3])
  stack = [4]
  pop 4: lá -> thêm (ans = [1, 3, 4])
[biên phải]
  nút 2: lấy (right = [2])
  nút 4: lá -> bỏ, dừng
[ghép] ans = [1, 3, 4, 2]


## 9. Code nộp LeetCode

Ta nộp **Cách 2**: `O(n)`, **không đệ quy** nên không lo `RecursionError` với cây sâu `10^4`. Trên LeetCode, `TreeNode`, `Optional`, `List` có sẵn, không cần định nghĩa lại.

In [19]:
class Solution:
    def boundaryOfBinaryTree(self, root: Optional[TreeNode]) -> List[int]:
        if not root.left and not root.right:
            return [root.val]
        ans = [root.val]

        node = root.left                      # 1) biên trái, từ trên xuống
        while node:
            if node.left or node.right:       # lá không thuộc biên
                ans.append(node.val)
            node = node.left if node.left else node.right

        stack = [root]                        # 2) lá, từ trái sang phải
        while stack:
            node = stack.pop()
            if not node.left and not node.right:
                ans.append(node.val)
            else:
                if node.right:
                    stack.append(node.right)
                if node.left:
                    stack.append(node.left)

        right = []                            # 3) biên phải, thu rồi đảo
        node = root.right
        while node:
            if node.left or node.right:
                right.append(node.val)
            node = node.right if node.right else node.left
        ans.extend(reversed(right))
        return ans


for vals, expected in examples:
    assert Solution().boundaryOfBinaryTree(build_tree(vals)) == expected
print("Code nộp đúng trên 2 ví dụ đề")

Code nộp đúng trên 2 ví dụ đề


## 10. Vì sao thuật toán đúng

**Ba phần phủ đúng tập biên, không trùng, không sót.**

- **Biên trái:** theo đề, nó là chuỗi `root.left = b1, b2, ...` trong đó `b(k+1)` là con trái của `bk` nếu có, nếu không thì con phải; chuỗi dừng ở lá (lá không thuộc biên trái). Vòng `while` đi **đúng** theo định nghĩa đệ quy này; mỗi nút không phải lá gặp trên đường đều được thêm. Cây hữu hạn nên vòng dừng.
- **Biên phải:** đối xứng. Thu từ trên xuống, nên đảo lại để có thứ tự từ dưới lên.
- **Lá:** duyệt tiền tự (cha, trái, phải) bằng ngăn xếp; lấy một nút ra rồi đẩy **phải trước, trái sau** thì nút trái được lấy ra trước nút phải, tức là **trái trước phải**. Mọi lá được lấy đúng một lần, theo thứ tự trái → phải.
- **Không trùng:** biên trái nằm trong cây con trái của gốc, biên phải nằm trong cây con phải, hai tập không giao nhau; cả hai **chỉ chứa nút không phải lá** nên không giao với tập lá; gốc không phải lá và không nằm trong cây con nào nên không giao với ba tập kia (trường hợp gốc là lá: trả `[root.val]` riêng).

**Vì sao Cách 4 (một lượt) cũng đúng.** Cờ `lb` của một nút là `True` đúng khi nút đó thuộc đường biên trái (quy nạp: con trái của nút biên trái luôn thuộc biên, con phải thuộc biên khi cha không có con trái). Tiền tự thăm các nút biên trái theo thứ tự trên xuống. Một lá chỉ có thể được thăm **sau** mọi nút biên trái ở trên nó (vì biên trái luôn ưu tiên đi trái nên không có nhánh nào treo bên trái đường biên), và mọi nút biên trái **dưới** một lá nào đó nằm trong cùng một nhánh nên được thăm trước lá đó. Ngược lại, hậu tự ghi nút biên phải sau khi đã thăm **toàn bộ** cây con của nó (mọi lá ở dưới và bên trái của nó) và theo thứ tự sâu → nông, tức đúng thứ tự đảo.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space (không tính kết quả) | Ghi chú |
|---|---|---|---|
| 1. Kiểm tra từng nút bằng đường đi | `O(n · h)` | `O(h)` | cây đường thẳng: `O(n²)` |
| 2. Ba vòng `while` + ngăn xếp (nộp) | `O(n)` | `O(h)` | không đệ quy |
| 3. Hàm đệ quy theo vai trò `i` | `O(n)` | `O(h)` | cần độ sâu đệ quy `h` |
| 4. Một lượt DFS với cờ `lb`, `rb` | `O(n)` | `O(h)` | cần độ sâu đệ quy `h` |

`n` là số nút, `h` là chiều cao cây (`log n` nếu cân bằng, `n` nếu là đường thẳng). Mọi cách tối ưu đều phải **đụng tới mọi lá** nên `O(n)` là cận dưới (cây có thể gồm toàn lá ở đáy). Về bộ nhớ, riêng danh sách kết quả có thể dài tới `O(n)`.

## 12. Bộ test

### 12.1. Ví dụ đề, biên và các cây tự dựng (đã tính tay)

Mỗi cây nhỏ dưới đây có kết quả **tính tay theo định nghĩa**:

| Cây (`list`) | Mô tả | Kết quả tay |
|---|---|---|
| `[1]` | một nút | `[1]` |
| `[1,2]` | gốc chỉ có con trái là lá | `[1,2]` |
| `[1,None,2]` | gốc chỉ có con phải là lá | `[1,2]` |
| `[1,2,3]` | hai lá | `[1,2,3]` |
| `[1,2,None,3]` | đường thẳng trái | `[1,2,3]` |
| `[1,None,2,None,3]` | đường thẳng phải | `[1,3,2]` |
| `[1,2,3,4,5,6,7]` | cây đầy đủ 3 tầng | `[1,2,4,5,6,7,3]` |
| `[1,2,3,None,4,5]` | biên rẽ sang con còn lại | `[1,2,4,5,3]` |
| `[1,2,None,None,3,4,5]` | biên trái rẽ phải rồi đi tiếp | `[1,2,3,4,5]` |
| `[1,2,3,None,4,None,None,5,6]` | biên trái rẽ phải; con phải của gốc là lá | `[1,2,4,5,6,3]` |
| `[1,1,1]` | giá trị trùng nhau | `[1,1,1]` |
| `[-1000,1000,-1000]` | giá trị biên | `[-1000,1000,-1000]` |

In [20]:
hand_cases = [
    ([1], [1]),
    ([1, 2], [1, 2]),
    ([1, None, 2], [1, 2]),
    ([1, 2, 3], [1, 2, 3]),
    ([1, 2, None, 3], [1, 2, 3]),
    ([1, None, 2, None, 3], [1, 3, 2]),
    ([1, 2, 3, 4, 5, 6, 7], [1, 2, 4, 5, 6, 7, 3]),
    ([1, 2, 3, None, 4, 5], [1, 2, 4, 5, 3]),
    ([1, 2, None, None, 3, 4, 5], [1, 2, 3, 4, 5]),
    ([1, 2, 3, None, 4, None, None, 5, 6], [1, 2, 4, 5, 6, 3]),
    ([1, 1, 1], [1, 1, 1]),
    ([-1000, 1000, -1000], [-1000, 1000, -1000]),
]
all_cases = examples + hand_cases

solvers = {
    "Cách 1 vét cạn": boundary_bruteforce,
    "Cách 2 ba vòng while": Solution2().boundaryOfBinaryTree,
    "Cách 3 vai trò i": Solution3().boundaryOfBinaryTree,
    "Cách 4 một lượt": Solution4().boundaryOfBinaryTree,
    "Code nộp": Solution().boundaryOfBinaryTree,
}

for vals, expected in all_cases:
    for name, f in solvers.items():
        got = f(build_tree(vals))
        assert got == expected, (name, vals, got, expected)
print("Tất cả", len(all_cases), "cây nhỏ đều đúng với mọi cách")

Tất cả 14 cây nhỏ đều đúng với mọi cách


### 12.2. Phản ví dụ cho các cách làm sai

Một cách làm "nghe có vẻ hợp lý" nhưng sai: **lấy nút đầu và nút cuối của mỗi tầng** làm biên trái/phải (kèm lá). Với cây có nhánh trái là một lá còn nhánh phải đi sâu, nút đầu của tầng sâu lại **không** thuộc biên trái.

In [21]:
def wrong_level_edges(root):
    """SAI: lấy nút đầu / cuối của mỗi tầng làm biên."""
    levels = []
    q = deque([root])
    while q:
        level = []
        for _ in range(len(q)):
            node = q.popleft()
            level.append(node)
            if node.left:
                q.append(node.left)
            if node.right:
                q.append(node.right)
        levels.append(level)

    def leaf(x):
        return x.left is None and x.right is None

    left = [lv[0].val for lv in levels[1:] if not leaf(lv[0])]
    right = [lv[-1].val for lv in levels[1:] if not leaf(lv[-1])]
    leaves = []
    st = [root]
    while st:
        x = st.pop()
        if leaf(x):
            leaves.append(x.val)
        else:
            if x.right:
                st.append(x.right)
            if x.left:
                st.append(x.left)
    return [root.val] + left + leaves + right[::-1]


tricky = [1, 2, 3, None, None, 4, None, 5]
draw(build_tree(tricky))
print("sai :", wrong_level_edges(build_tree(tricky)))
print("đúng:", Solution().boundaryOfBinaryTree(build_tree(tricky)))
assert Solution().boundaryOfBinaryTree(build_tree(tricky)) == [1, 2, 5, 4, 3]
assert wrong_level_edges(build_tree(tricky)) != [1, 2, 5, 4, 3]

1
├─L:2
└─R:3
   └─L:4
      └─L:5
sai : [1, 4, 2, 5, 4, 3]
đúng: [1, 2, 5, 4, 3]


Vì sao sai: biên trái ở đây **rỗng** (con trái của gốc là lá `2`), nhưng nút đầu của tầng 2 là `4` (thuộc nhánh phải) bị nhận nhầm là biên trái, và `4` xuất hiện hai lần.

### 12.3. Đối chiếu ngẫu nhiên có seed, với oracle vét cạn

Oracle là **Cách 1** (kiểm tra từng nút bằng đường đi, hoàn toàn khác cách đi dọc biên). Ta sinh cây ngẫu nhiên đúng ràng buộc: kích thước `1..40`, giá trị trong `[-1000, 1000]` (có thể trùng), nhiều hình dạng nhờ tham số `chain_bias` (kéo cây về dạng đường dài/rẽ nhánh).

In [22]:
def random_tree(n, rng, lo=-1000, hi=1000, chain_bias=0.0):
    """Cây ngẫu nhiên đúng n nút. chain_bias lớn -> cây dài, ít phân nhánh."""
    root = TreeNode(rng.randint(lo, hi))
    slots = [(root, "left"), (root, "right")]
    for _ in range(n - 1):
        if chain_bias and rng.random() < chain_bias:
            idx = len(slots) - 1 - rng.randint(0, min(1, len(slots) - 1))
        else:
            idx = rng.randrange(len(slots))
        slots[idx], slots[-1] = slots[-1], slots[idx]
        node, side = slots.pop()
        new = TreeNode(rng.randint(lo, hi))
        setattr(node, side, new)
        slots.append((new, "left"))
        slots.append((new, "right"))
    return root


def count_nodes(root):
    return 0 if root is None else 1 + count_nodes(root.left) + count_nodes(root.right)


rng = random.Random(545)
checked = 0
for _ in range(2000):
    n = rng.randint(1, 40)
    root = random_tree(n, rng, chain_bias=rng.choice([0, 0, 0.5, 0.9]))
    assert count_nodes(root) == n
    expected = boundary_bruteforce(root)
    assert Solution2().boundaryOfBinaryTree(root) == expected
    assert Solution3().boundaryOfBinaryTree(root) == expected
    assert Solution4().boundaryOfBinaryTree(root) == expected
    assert Solution().boundaryOfBinaryTree(root) == expected
    assert 1 <= len(expected) <= n and expected[0] == root.val
    checked += 1
print("Đã đối chiếu", checked, "cây ngẫu nhiên (seed 545)")

Đã đối chiếu 2000 cây ngẫu nhiên (seed 545)


### 12.4. Cây lớn và cây rất sâu (giới hạn của đề)

Ba hình dạng cực đoan với `n = 10^4`: đường thẳng trái, đường thẳng phải, zig-zag; cộng một cây ngẫu nhiên lớn. Với cây sâu `10^4`, các cách **đệ quy** (Cách 3, 4) cần nới giới hạn đệ quy; code nộp (Cách 2) **không cần**. Oracle vét cạn `O(n·h)` quá chậm cho đường thẳng, nên ta so Cách 2/3/4 với nhau và với kết quả **biết trước bằng công thức**:

- Đường thẳng trái `n` nút giá trị `1..n` (mỗi nút là con trái của nút trước): biên trái là `2..n-1`, lá là `n`, biên phải rỗng ⇒ `[1, 2, ..., n]`.
- Đường thẳng phải `n` nút: biên trái rỗng, lá `n`, biên phải `2..n-1` đảo ⇒ `[1, n, n-1, ..., 2]`.

In [23]:
def chain(n, side):
    root = TreeNode(1)
    cur = root
    for v in range(2, n + 1):
        new = TreeNode(v)
        setattr(cur, side, new)
        cur = new
    return root


def zigzag(n):
    root = TreeNode(1)
    cur = root
    for v in range(2, n + 1):
        new = TreeNode(v)
        if v % 2 == 0:
            cur.left = new
        else:
            cur.right = new
        cur = new
    return root


N = 10 ** 4
sys.setrecursionlimit(50000)          # chỉ để thử các cách đệ quy trên cây sâu

left_chain = chain(N, "left")
right_chain = chain(N, "right")
assert Solution().boundaryOfBinaryTree(left_chain) == list(range(1, N + 1))
assert Solution().boundaryOfBinaryTree(right_chain) == [1] + list(range(N, 1, -1))

big_cases = {
    "đường thẳng trái": left_chain,
    "đường thẳng phải": right_chain,
    "zig-zag": zigzag(N),
    "ngẫu nhiên lớn": random_tree(N, random.Random(5450), chain_bias=0.3),
}
for name, tree in big_cases.items():
    ref = Solution().boundaryOfBinaryTree(tree)
    assert Solution2().boundaryOfBinaryTree(tree) == ref
    assert Solution3().boundaryOfBinaryTree(tree) == ref
    assert Solution4().boundaryOfBinaryTree(tree) == ref
    print(f"{name:<18} n = {N}: biên có {len(ref)} nút")

đường thẳng trái   n = 10000: biên có 10000 nút


đường thẳng phải   n = 10000: biên có 10000 nút
zig-zag            n = 10000: biên có 10000 nút
ngẫu nhiên lớn     n = 10000: biên có 2934 nút


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Tính lá hai lần (và lặp nút gốc khi cây một nút)

Nếu biên trái/phải **cũng thu lá**, rồi phần "lá" lại thu lá lần nữa, nút lá xuất hiện hai lần. Tương tự, cây chỉ có một nút, nếu không xử lý riêng, gốc vừa là "gốc" vừa bị coi là "lá".

In [24]:
def sai_lap_la(root):
    ans = [root.val]
    node = root.left
    while node:                          # SAI: thu cả lá vào biên trái
        ans.append(node.val)
        node = node.left if node.left else node.right
    stack = [root]
    while stack:
        node = stack.pop()
        if not node.left and not node.right:
            ans.append(node.val)
        else:
            if node.right:
                stack.append(node.right)
            if node.left:
                stack.append(node.left)
    return ans


print("sai :", sai_lap_la(build_tree([1, 2, 3])))              # lá 2 xuất hiện hai lần
print("đúng:", Solution().boundaryOfBinaryTree(build_tree([1, 2, 3])))
print("cây một nút, sai :", sai_lap_la(build_tree([7])))       # gốc lặp
print("cây một nút, đúng:", Solution().boundaryOfBinaryTree(build_tree([7])))

sai : [1, 2, 2, 3]
đúng: [1, 2, 3]
cây một nút, sai : [7, 7]
cây một nút, đúng: [7]


**Vì sao sai:** biên trái/phải **không chứa lá** theo đề ("lá trái nhất không thuộc biên trái"). Lá chỉ được tính ở phần **lá**. **Sửa:** chỉ thêm nút khi `node.left or node.right`; và xử lý riêng cây một nút (`return [root.val]`).

### Lỗi 2: Biên trái chỉ đi theo `.left`, quên rẽ sang con phải

Quy tắc đúng: nếu nút không có con trái nhưng có con phải thì **đi tiếp sang con phải**. Quên điều này làm biên trái bị cắt ngắn.

In [25]:
def sai_chi_di_trai(root):
    if not root.left and not root.right:
        return [root.val]
    ans = [root.val]
    node = root.left
    while node:
        if node.left or node.right:
            ans.append(node.val)
        node = node.left                     # SAI: không rẽ sang node.right
    stack = [root]
    while stack:
        node = stack.pop()
        if not node.left and not node.right:
            ans.append(node.val)
        else:
            if node.right:
                stack.append(node.right)
            if node.left:
                stack.append(node.left)
    right = []
    node = root.right
    while node:
        if node.left or node.right:
            right.append(node.val)
        node = node.right if node.right else node.left
    ans.extend(reversed(right))
    return ans


t = [1, 2, 3, None, 4, None, None, 5, 6]
draw(build_tree(t))
print("sai :", sai_chi_di_trai(build_tree(t)))                    # thiếu nút 4 của biên trái
print("đúng:", Solution().boundaryOfBinaryTree(build_tree(t)))    # [1, 2, 4, 5, 6, 3]

1
├─L:2
│  └─R:4
│     ├─L:5
│     └─R:6
└─R:3
sai : [1, 2, 5, 6, 3]
đúng: [1, 2, 4, 5, 6, 3]


**Vì sao sai:** nút `2` không có con trái, chỉ có con phải `4`; theo đề `4` thuộc biên trái (và `4` không phải lá vì còn `5, 6`). **Sửa:** `node = node.left if node.left else node.right`. Biên phải làm đối xứng.

### Lỗi 3: Quên đảo biên phải

Biên phải được **đi từ trên xuống** nên phải **đảo** trước khi nối; nếu không thứ tự sai (ví dụ `[..., 3, 6]` thay vì `[..., 6, 3]`).

In [26]:
def sai_khong_dao(root):
    correct = Solution().boundaryOfBinaryTree(root)
    # mô phỏng lỗi: nối biên phải KHÔNG đảo
    n_right = 0
    node = root.right
    while node:
        if node.left or node.right:
            n_right += 1
        node = node.right if node.right else node.left
    if n_right <= 1:
        return correct
    head, tail = correct[:-n_right], correct[-n_right:]
    return head + tail[::-1]


print("sai :", sai_khong_dao(tree2))                       # [1, 2, 4, 7, 8, 9, 10, 3, 6]
print("đúng:", Solution().boundaryOfBinaryTree(tree2))     # [1, 2, 4, 7, 8, 9, 10, 6, 3]
assert sai_khong_dao(tree2) != Solution().boundaryOfBinaryTree(tree2)

sai : [1, 2, 4, 7, 8, 9, 10, 3, 6]
đúng: [1, 2, 4, 7, 8, 9, 10, 6, 3]


**Vì sao sai:** đề yêu cầu biên phải theo **thứ tự ngược** (đọc từ dưới lên khi đi hết vòng). **Sửa:** `ans.extend(reversed(right))`, hoặc ghi ở thời điểm hậu tự như Cách 4.

### Lỗi 4: Coi nút chỉ có một con là lá (dùng `or` thay vì `and`)

Lá là nút **không có con nào**: `not node.left and not node.right`. Viết `not node.left or not node.right` (hoặc chỉ kiểm tra một phía) coi nhầm nút chỉ có một con là lá.

In [27]:
def sai_la_dung_or(root):
    if not root.left and not root.right:
        return [root.val]
    ans = [root.val]
    node = root.left
    while node:
        if node.left or node.right:
            ans.append(node.val)
        node = node.left if node.left else node.right
    stack = [root]
    while stack:
        node = stack.pop()
        if not node.left or not node.right:          # SAI: or
            ans.append(node.val)
        else:
            stack.append(node.right)
            stack.append(node.left)
    right = []
    node = root.right
    while node:
        if node.left or node.right:
            right.append(node.val)
        node = node.right if node.right else node.left
    ans.extend(reversed(right))
    return ans


t = [1, 2, 3, 4, None, 5, 6]
print("sai :", sai_la_dung_or(build_tree(t)))               # 2 (chỉ có con trái 4) bị coi là lá
print("đúng:", Solution().boundaryOfBinaryTree(build_tree(t)))
assert sai_la_dung_or(build_tree(t)) != Solution().boundaryOfBinaryTree(build_tree(t))

sai : [1, 2, 2, 5, 6, 3]
đúng: [1, 2, 4, 5, 6, 3]


**Vì sao sai:** nút `2` có con trái `4` nên **không** là lá; nhưng kiểm tra `not node.right` đúng với `2` nên nó bị thu **thêm một lần nữa** như lá (kết quả có hai số `2`), và nút con `4` của nó không được đẩy vào ngăn xếp nên bị mất. **Sửa:** `not node.left and not node.right`.

### Lỗi 5: Đệ quy trên cây rất sâu

Cách 3 và 4 dùng đệ quy; với cây là đường thẳng dài `10^4`, Python mặc định (giới hạn khoảng `1000`) có thể báo `RecursionError`. Nếu thích đệ quy, hãy thêm `sys.setrecursionlimit(...)`; nếu muốn chắc ăn thì dùng vòng `while` + ngăn xếp như Cách 2.

## 14. Mẹo & nhận dạng pattern

- **Dấu hiệu:** "boundary", "viền", "ngược chiều kim đồng hồ", "left view / right view" ⇒ chia bài toán thành các **phần độc lập** (trái, đáy, phải) rồi **nối**.
- **Đường đi xuống một nhánh** (biên trái/phải) chỉ là **một vòng `while`**, không cần đệ quy.
- **Lá từ trái sang phải** ⇒ duyệt **tiền tự** (hoặc trung tự, hậu tự cho cùng thứ tự lá), ngăn xếp: đẩy **phải trước, trái sau**.
- **Cần đảo thứ tự "từ dưới lên"** ⇒ hoặc `reversed`, hoặc ghi ở thời điểm **hậu tự**.
- **Loại trừ trùng lặp ngay từ định nghĩa:** biên không chứa lá; gốc xử lý riêng. Khi gặp đề "gộp nhiều nhóm", hãy kiểm tra các nhóm **có giao nhau không**.
- **Giá trị trùng nhau → làm việc với nút, không với giá trị.** Không dùng `set` giá trị để khử trùng.
- **Cây sâu `10^4` trong Python:** ưu tiên vòng lặp + ngăn xếp.
- **Vẽ cây ra giấy (hoặc `draw`)** rồi đi tay một vòng trước khi viết code.

## 15. Biến thể

1. **Biên theo chiều kim đồng hồ:** gốc, biên phải (từ trên xuống), lá từ **phải sang trái**, biên trái từ **dưới lên**.
2. **Đếm số nút biên** thay vì liệt kê: dùng đúng ba phần, chỉ đếm.
3. **Trả về nút** (không phải giá trị) để xử lý trường hợp giá trị trùng.
4. **Mẹo đối xứng:** biên phải của cây là biên trái của **cây đã lật gương** (đổi chỗ trái/phải) nhưng đảo thứ tự; nhờ vậy chỉ cần viết một hàm "biên trái".
5. **Left view / Right view of Binary Tree** (nút đầu/cuối của mỗi tầng): bài **khác**, dựa trên tầng (BFS), đừng nhầm với biên.
6. **Biên của cây N-phân** hoặc **đồ thị phẳng**: ý tưởng tổng quát là "đi dọc mép".

In [28]:
def boundary_clockwise(root):
    """Biến thể 1: theo chiều kim đồng hồ = đảo toàn bộ kết quả ngược chiều, giữ gốc đầu."""
    ccw = Solution().boundaryOfBinaryTree(root)
    return [ccw[0]] + ccw[1:][::-1]


def count_boundary(root):
    """Biến thể 2: số nút biên (không dựng kết quả)."""
    return len(Solution().boundaryOfBinaryTree(root))


def mirror(node):
    """Lật gương: đổi chỗ trái/phải (dựng cây mới)."""
    if node is None:
        return None
    return TreeNode(node.val, mirror(node.right), mirror(node.left))


def boundary_by_mirror(root):
    """Biến thể 4: biên của cây = biên của cây lật gương, đảo phần giữa (chỉ để minh họa)."""
    if not root.left and not root.right:
        return [root.val]
    ans = [root.val]
    # biên trái bình thường
    node = root.left
    while node:
        if node.left or node.right:
            ans.append(node.val)
        node = node.left if node.left else node.right
    # lá
    stack = [root]
    while stack:
        node = stack.pop()
        if not node.left and not node.right:
            ans.append(node.val)
        else:
            if node.right:
                stack.append(node.right)
            if node.left:
                stack.append(node.left)
    # biên phải = "biên trái" của cây gương ở phía con (gốc gương có con trái = root.right)
    m = mirror(root)
    node = m.left
    right = []
    while node:
        if node.left or node.right:
            right.append(node.val)
        node = node.left if node.left else node.right
    ans.extend(reversed(right))
    return ans


print(boundary_clockwise(tree2))        # [1, 3, 6, 10, 9, 8, 7, 4, 2]
print(count_boundary(tree2))            # 9
print(boundary_by_mirror(tree2) == Solution().boundaryOfBinaryTree(tree2))   # True

[1, 3, 6, 10, 9, 8, 7, 4, 2]
9
True


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Kiểm tra từng nút bằng đường đi | "Hỏi" từng nút có thuộc biên trái/phải/lá không, dựa vào `path` | `O(n·h)` | `O(h)` | Làm oracle để đối chiếu; không nộp |
| 2. Ba vòng `while` + ngăn xếp | Đi dọc biên trái, thu lá, đi dọc biên phải rồi đảo | `O(n)` | `O(h)` | **Nộp bài**: không đệ quy, an toàn cây sâu |
| 3. Hàm đệ quy theo vai trò `i` | Cùng ý ba phần nhưng viết một hàm đệ quy `0/1/2` | `O(n)` | `O(h)` đệ quy | Gọn, dễ sao sang ngôn ngữ khác (lời giải chính thức) |
| 4. Một lượt DFS với `lb`, `rb` | Truyền cờ xuống; biên trái ghi lúc vào, biên phải ghi lúc ra | `O(n)` | `O(h)` đệ quy | Muốn một lượt duyệt; khó đọc hơn |

Ghi chú: Cách 2 và 3 cùng ý tưởng "ba phần" (khác nhau ở **cách viết**: lặp vs đệ quy); Cách 1 và Cách 4 là hai cách nhìn khác hẳn (hỏi từng nút / một lượt có cờ).

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `tree_to_list(build_tree([1, None, 2, 3]))`; `Solution().boundaryOfBinaryTree(build_tree([5]))`; biên của `[1, None, 2, None, 3]`; biên của `[1, 2, 3]`; và `is_leaf(TreeNode(4, TreeNode(5), None))`.

**Bài 2 (tính tay).** Với cây `[1, 2, 3, 4, 5, 6, 7, 8, 9]`, vẽ cây (hoặc `draw`) rồi liệt kê **biên trái**, **các lá**, **biên phải (trước khi đảo)** và kết quả cuối.

**Bài 3 (điền TODO).** Hoàn thành `left_boundary_todo(root)` chỉ trả về **biên trái** (danh sách giá trị, từ trên xuống, không chứa lá, bắt đầu từ `root.left`).

In [29]:
def left_boundary_todo(root):
    # TODO 1: ans = []; node = root.left
    # TODO 2: while node: nếu node KHÔNG phải lá thì ans.append(node.val)
    # TODO 3: node = node.left nếu có, ngược lại node.right
    # TODO 4: return ans
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên chưa đúng).

**Bài 4 (tự viết lại).** Không nhìn code mẫu, viết `boundary_oneshot(root)` theo **Cách 4** (một lượt DFS với cờ `lb`, `rb`; tiền tự cho biên trái và lá, hậu tự cho biên phải).

**Bài 5 (biến thể / mini challenge).** (a) Viết `boundary_clockwise2(root)` trả về biên **theo chiều kim đồng hồ** nhưng **không** gọi lại `Solution` (dựng trực tiếp: gốc, biên phải từ trên xuống, lá từ **phải sang trái**, biên trái từ **dưới lên**). (b) Viết `boundary_nodes_count_by_part(root)` trả về bộ ba `(số nút biên trái, số lá, số nút biên phải)`; với cây một nút trả `(0, 0, 0)`.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[1, None, 2, 3]`; `[5]`; `[1, 3, 2]`; `[1, 2, 3]`; `False` (nút `4` có con trái nên không phải lá). Bài 3: bám sát mục 6 và vòng `while` của Cách 2.
- **Mức 2 (vừa):** Bài 2: vẽ cây: gốc `1`; con `2`, `3`; con của `2` là `4`, `5`; con của `3` là `6`, `7`; con của `4` là `8`, `9`. Biên trái: `2`, `4` (đi trái; `8` là lá nên bỏ). Lá: `8, 9, 5, 6, 7`. Biên phải: `3` (rẽ phải sang `7` là lá, bỏ) nên `[3]`. Kết quả `[1,2,4,8,9,5,6,7,3]`. Bài 4: `dfs(node, lb, rb)` ghi khi vào nếu `leaf or lb`; sau hai lời gọi con, ghi nếu `rb and not leaf`.
- **Mức 3 (sâu):** Bài 5(a): giữ ba list: `right_top_down` (đi theo biên phải), `leaves` (lá theo thứ tự **phải trước trái**: lấy ra bằng ngăn xếp đẩy **trái trước, phải sau**), `left_top_down` (biên trái); ghép `[root] + right_top_down + leaves + reversed(left_top_down)`. Bài 5(b): đếm giống như Cách 2 nhưng chỉ cộng biến đếm.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [30]:
# Bài 1
assert tree_to_list(build_tree([1, None, 2, 3])) == [1, None, 2, 3]
assert Solution().boundaryOfBinaryTree(build_tree([5])) == [5]
assert Solution().boundaryOfBinaryTree(build_tree([1, None, 2, None, 3])) == [1, 3, 2]
assert Solution().boundaryOfBinaryTree(build_tree([1, 2, 3])) == [1, 2, 3]
assert is_leaf(TreeNode(4, TreeNode(5), None)) is False

# Bài 2
tree_b2 = build_tree([1, 2, 3, 4, 5, 6, 7, 8, 9])
draw(tree_b2)
assert Solution().boundaryOfBinaryTree(tree_b2) == [1, 2, 4, 8, 9, 5, 6, 7, 3]
assert boundary_bruteforce(tree_b2) == [1, 2, 4, 8, 9, 5, 6, 7, 3]


# Bài 3
def left_boundary_ans(root):
    ans = []
    node = root.left
    while node:
        if node.left or node.right:
            ans.append(node.val)
        node = node.left if node.left else node.right
    return ans


assert left_boundary_ans(tree_b2) == [2, 4]
assert left_boundary_ans(build_tree(ex1)) == []
assert left_boundary_ans(tree2) == [2]
assert left_boundary_ans(build_tree([1, 2, 3, None, 4, None, None, 5, 6])) == [2, 4]


# Bài 4
def boundary_oneshot(root):
    ans = [root.val]
    if not root.left and not root.right:
        return ans

    def dfs(node, lb, rb):
        if node is None:
            return
        leaf = not node.left and not node.right
        if leaf or lb:
            ans.append(node.val)
        dfs(node.left, lb, rb and node.right is None)
        dfs(node.right, lb and node.left is None, rb)
        if rb and not leaf:
            ans.append(node.val)

    dfs(root.left, True, False)
    dfs(root.right, False, True)
    return ans


for vals, expected in all_cases:
    assert boundary_oneshot(build_tree(vals)) == expected


# Bài 5
def boundary_clockwise2(root):
    if not root.left and not root.right:
        return [root.val]
    right_top_down = []
    node = root.right
    while node:
        if node.left or node.right:
            right_top_down.append(node.val)
        node = node.right if node.right else node.left
    leaves = []
    stack = [root]
    while stack:
        node = stack.pop()
        if not node.left and not node.right:
            leaves.append(node.val)
        else:
            if node.left:
                stack.append(node.left)
            if node.right:
                stack.append(node.right)
    left_top_down = left_boundary_ans(root)
    return [root.val] + right_top_down + leaves + left_top_down[::-1]


def boundary_nodes_count_by_part(root):
    if not root.left and not root.right:
        return (0, 0, 0)
    n_left = len(left_boundary_ans(root))
    n_right = 0
    node = root.right
    while node:
        if node.left or node.right:
            n_right += 1
        node = node.right if node.right else node.left
    n_leaves = 0
    stack = [root]
    while stack:
        node = stack.pop()
        if not node.left and not node.right:
            n_leaves += 1
        else:
            if node.left:
                stack.append(node.left)
            if node.right:
                stack.append(node.right)
    return (n_left, n_leaves, n_right)


rng2 = random.Random(5451)
for _ in range(300):
    r = random_tree(rng2.randint(1, 30), rng2, chain_bias=rng2.choice([0, 0.7]))
    assert boundary_clockwise2(r) == boundary_clockwise(r)
    a, b, c = boundary_nodes_count_by_part(r)
    assert (a + b + c + 1 == count_boundary(r)) or (a, b, c) == (0, 0, 0)
assert boundary_clockwise2(tree2) == [1, 3, 6, 10, 9, 8, 7, 4, 2]
assert boundary_nodes_count_by_part(tree2) == (1, 5, 2)
assert boundary_nodes_count_by_part(build_tree([1])) == (0, 0, 0)
print("Tất cả đáp án bài tập đều đúng.")

1
├─L:2
│  ├─L:4
│  │  ├─L:8
│  │  └─R:9
│  └─R:5
└─R:3
   ├─L:6
   └─R:7
Tất cả đáp án bài tập đều đúng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (ba phần: biên trái, lá, biên phải đảo; không đệ quy)

```python
class Solution:
    def boundaryOfBinaryTree(self, root: Optional[TreeNode]) -> List[int]:
        if not root.left and not root.right:
            return [root.val]
        ans = [root.val]

        node = root.left                           # biên trái (trên -> xuống)
        while node:
            if node.left or node.right:            # lá KHÔNG thuộc biên
                ans.append(node.val)
            node = node.left if node.left else node.right

        stack = [root]                             # lá, trái -> phải
        while stack:
            node = stack.pop()
            if not node.left and not node.right:
                ans.append(node.val)
            else:
                if node.right:
                    stack.append(node.right)       # phải trước
                if node.left:
                    stack.append(node.left)        # trái sau (pop ra trước)

        right = []                                 # biên phải (trên -> xuống) rồi đảo
        node = root.right
        while node:
            if node.left or node.right:
                right.append(node.val)
            node = node.right if node.right else node.left
        ans.extend(reversed(right))
        return ans
```

**Checklist**

- [ ] Cây **một nút** trả `[root.val]` (gốc không phải lá, không lặp)?
- [ ] Biên trái ưu tiên **con trái, không có thì con phải**; biên phải ngược lại?
- [ ] **Lá** không nằm trong biên trái/phải (chỉ ở nhóm lá); lá = **không có con nào** (`and`, không phải `or`)?
- [ ] Biên phải đã **đảo** (hoặc ghi ở hậu tự)?
- [ ] Lá theo thứ tự **trái sang phải** (đẩy phải trước, trái sau khi dùng ngăn xếp)?
- [ ] Làm việc với **nút**, không khử trùng theo giá trị (giá trị có thể trùng)?
- [ ] Cây sâu `10^4`: dùng vòng lặp + ngăn xếp (hoặc nới `setrecursionlimit` nếu dùng đệ quy)?
- [ ] Đã thử: `[1]`, `[1,2]`, `[1,None,2]`, đường thẳng trái/phải, ví dụ 1 (`[1,3,4,2]`) và ví dụ 2?
- [ ] Skeleton đúng `class Solution` / `def boundaryOfBinaryTree(self, root: Optional[TreeNode]) -> List[int]`?